In [115]:
"""
Linear Regression Pipeline — WV Opioid Data
Target: LA_Opioid_Rate
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score


In [116]:
df = pd.read_csv("../data/west_virginia_opioid_data_clean.csv")
df

,Year,FIPS_Code,State,County,Labor_Force_Participation_Rate,Unemployment_Rate,Median_Household_Income,Poverty_Percent_All_Ages,Poverty_Percent_Age_0_17,Pct_Never_Married,Pct_Divorced,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,Rural_Urban_Continuum_Code_2023,LA_Opioid_Rate,LA_Opioid_Prscrbng_Rate_1Y_Chg
0,2019,54001,West Virginia,Barbour,52.6,7.6,38459,20.8,30.8,48.3,8.0,10.9,47.3,15.6,9,4.89,-0.68
1,2020,54001,West Virginia,Barbour,53.0,8.4,38906,21.0,32.7,47.1,10.0,8.7,50.6,13.9,9,5.50,0.61
2,2021,54001,West Virginia,Barbour,50.3,10.0,42260,20.8,30.5,47.1,10.2,8.6,53.1,13.0,9,6.42,0.92
3,2022,54001,West Virginia,Barbour,49.9,10.1,44341,22.0,32.5,48.1,10.8,8.5,54.0,11.8,9,6.87,0.45
4,2023,54001,West Virginia,Barbour,50.0,10.3,48347,20.8,29.1,45.8,10.3,8.0,53.4,12.2,9,5.41,-1.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,2019,54109,West Virginia,Wyoming,39.9,13.4,42332,22.4,29.4,59.7,8.4,11.4,48.7,9.3,8,0.36,0.15
271,2020,54109,West Virginia,Wyoming,41.0,11.6,44095,21.4,29.8,62.6,7.7,13.5,46.6,11.8,8,0.36,0.15
272,2021,54109,West Virginia,Wyoming,36.9,8.0,44630,25.3,31.2,55.8,8.0,14.0,46.7,11.6,8,0.51,0.15
273,2022,54109,West Virginia,Wyoming,37.3,6.4,44510,24.4,31.3,51.7,9.4,15.8,46.6,11.4,8,0.94,0.43


In [117]:
# --- 2. Sort and shift target ---
df_sorted = df.sort_values(['FIPS_Code', 'Year'])
df_sorted['target_next'] = df_sorted.groupby('FIPS_Code')['LA_Opioid_Prscrbng_Rate_1Y_Chg'].shift(-1)
df_sorted = df_sorted.dropna(subset=['target_next'])
df_sorted

,Year,FIPS_Code,State,County,Labor_Force_Participation_Rate,Unemployment_Rate,Median_Household_Income,Poverty_Percent_All_Ages,Poverty_Percent_Age_0_17,Pct_Never_Married,Pct_Divorced,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,Rural_Urban_Continuum_Code_2023,LA_Opioid_Rate,LA_Opioid_Prscrbng_Rate_1Y_Chg,target_next
0,2019,54001,West Virginia,Barbour,52.6,7.6,38459,20.8,30.8,48.3,8.0,10.9,47.3,15.6,9,4.89,-0.68,0.61
1,2020,54001,West Virginia,Barbour,53.0,8.4,38906,21.0,32.7,47.1,10.0,8.7,50.6,13.9,9,5.50,0.61,0.92
2,2021,54001,West Virginia,Barbour,50.3,10.0,42260,20.8,30.5,47.1,10.2,8.6,53.1,13.0,9,6.42,0.92,0.45
3,2022,54001,West Virginia,Barbour,49.9,10.1,44341,22.0,32.5,48.1,10.8,8.5,54.0,11.8,9,6.87,0.45,-1.46
5,2019,54003,West Virginia,Berkeley,65.4,6.0,62515,12.2,18.2,51.9,5.2,7.9,37.6,21.2,2,11.97,-0.69,-3.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,2022,54107,West Virginia,Wood,55.2,5.3,54350,14.8,19.3,49.0,7.7,6.7,35.3,22.0,3,3.85,-0.35,-0.49
270,2019,54109,West Virginia,Wyoming,39.9,13.4,42332,22.4,29.4,59.7,8.4,11.4,48.7,9.3,8,0.36,0.15,0.15
271,2020,54109,West Virginia,Wyoming,41.0,11.6,44095,21.4,29.8,62.6,7.7,13.5,46.6,11.8,8,0.36,0.15,0.15
272,2021,54109,West Virginia,Wyoming,36.9,8.0,44630,25.3,31.2,55.8,8.0,14.0,46.7,11.6,8,0.51,0.15,0.43


In [118]:
# --- 3. Select numeric features, explicitly excluding leaky columns ---
cols_to_exclude = ['LA_Opioid_Prscrbng_Rate_1Y_Chg', 'target_next', 'Year', 'FIPS_Code']
numeric_features = df_sorted.select_dtypes(include='number').columns.tolist()
numeric_features = [c for c in numeric_features if c not in cols_to_exclude]

X = df_sorted[numeric_features]
y = df_sorted['target_next']

In [119]:
X.isna().sum()

Labor_Force_Participation_Rate     0
Unemployment_Rate                  0
Median_Household_Income            0
Poverty_Percent_All_Ages           0
Poverty_Percent_Age_0_17           0
Pct_Never_Married                  0
Pct_Divorced                       0
Pct_Less_Than_HS                   0
Pct_HS_Grad                        0
Pct_Bachelors_Plus                 0
Rural_Urban_Continuum_Code_2023    0
LA_Opioid_Rate                     0
dtype: int64

In [120]:
# --- 5. Temporal train/test split (avoids data leakage across time) ---
unique_years = sorted(df_sorted['Year'].unique())
print(f"Years in dataset: {unique_years}")

# Use the last 20% of unique years as test, ensuring at least 1 year in each split
n_test_years = max(1, int(len(unique_years) * 0.2))
cutoff_year = unique_years[-(n_test_years + 1)]
print(f"Train years: up to and including {cutoff_year}")
print(f"Test years:  {unique_years[-n_test_years:]}")

train_mask = df_sorted['Year'] <= cutoff_year
test_mask  = df_sorted['Year'] > cutoff_year

X_train = X[train_mask]
X_test  = X[test_mask]
y_train = y[train_mask]
y_test  = y[test_mask]

Years in dataset: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Train years: up to and including 2021
Test years:  [np.int64(2022)]


In [121]:
# --- 6. Scale features (fit on train only) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [122]:
# --- 7. Ridge regression with expanded alpha grid and TimeSeriesSplit CV ---
alphas = np.logspace(-3, 4, 50)  # 0.001 to 10,000 across 50 values
tscv = TimeSeriesSplit(n_splits=5)

ridge_cv = RidgeCV(alphas=alphas, cv=tscv)
ridge_cv.fit(X_train_scaled, y_train)


,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.",array([1.0000...00000000e+04])
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error ` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination ` (:math:`R^2`) otherwise.",None
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.",TimeSeriesSpl...est_size=None)
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : use 'svd' if n_samples > n_features, otherwise use 'eigen' 'svd' : force use of singular value decomposition of X when X is dense, eigenvalue decomposition of X^T.X when X is sparse. 'eigen' : force computation via eigendecomposition of X.X^TThe 'auto' mode is the default and is intended to pick the cheaperoption of the two depending on the shape of the training data.",None
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",False
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets... versionadded:: 0.24",False


In [123]:
# --- 8. Predict ---
y_pred = ridge_cv.predict(X_test_scaled)

In [124]:
# --- 9. Evaluate ---
mse  = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)  # Same units as target
r2   = r2_score(y_test, y_pred)

print(f"Best alpha (L2): {ridge_cv.alpha_}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

Best alpha (L2): 10000.0
MSE:  4.4903
RMSE: 2.1190
R²:   0.0043


In [125]:
# --- 10. Feature importance ---
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': ridge_cv.coef_
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(coefficients)

                            Feature  Coefficient
3          Poverty_Percent_All_Ages     0.005323
2           Median_Household_Income    -0.005224
4          Poverty_Percent_Age_0_17     0.004861
6                      Pct_Divorced     0.003741
7                  Pct_Less_Than_HS     0.002763
1                 Unemployment_Rate     0.002638
5                 Pct_Never_Married    -0.002503
10  Rural_Urban_Continuum_Code_2023     0.001876
0    Labor_Force_Participation_Rate    -0.001707
8                       Pct_HS_Grad    -0.001120
11                   LA_Opioid_Rate     0.001006
9                Pct_Bachelors_Plus    -0.000460
